In [1]:
# The following code will only execute
# successfully when compression is complete

import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohabenloughmari/cortexmnist")

print("Path to dataset files:", path)

100%|██████████| 2.95G/2.95G [00:17<00:00, 182MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mohabenloughmari/cortexmnist/versions/1


In [2]:
print(path)

/root/.cache/kagglehub/datasets/mohabenloughmari/cortexmnist/versions/1


In [3]:
import torch

data_train= torch.load(f'{path}/cortex_mnist/train.pt')
data_val= torch.load(f'{path}/cortex_mnist/val.pt')
data_test= torch.load(f'{path}/cortex_mnist/test.pt')


In [4]:
!git clone https://github.com/MohamedBenloughmari/Cortex-Image-Reconstruction.git

Cloning into 'Cortex-Image-Reconstruction'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 17 (delta 5), reused 13 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 15.00 KiB | 1.88 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [5]:
%cd Cortex-Image-Reconstruction
from ConvLSTM import ConvLSTM
import torch.nn as nn

/content/Cortex-Image-Reconstruction


In [6]:
print(data_test.keys())

dict_keys(['spikes_on', 'spikes_off', 'images'])


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

#class SpikesDataset(Dataset):
#    def __init__(self, data_dict):
#        self.spikes_on  = torch.tensor(data_dict['spikes_on'],  dtype=torch.int32)
#        self.spikes_off = torch.tensor(data_dict['spikes_off'], dtype=torch.float32)
#        self.images     = torch.tensor(data_dict['images'],     dtype=torch.long)
#        self.spikes_on.unsqueeze_(1)
#        self.spikes_off.unsqueeze_(1)
#        self.images.unsqueeze_(1)
#    def __len__(self):
#        return len(self.images)
#    def __getitem__(self, idx):
#        return {
#            'spikes_on':  self.spikes_on[idx],
#            'spikes_off': self.spikes_off[idx],
#            'image':      self.images[idx]
#        }

class SpikesDataset(Dataset):
    def __init__(self, data_dict):
        self.spikes_on = data_dict['spikes_on']
        self.images    = data_dict['images']

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        spikes_on = self.spikes_on[idx].float().unsqueeze(1)  # no clone
        image     = self.images[idx].float().unsqueeze(0)
        return {'spikes_on': spikes_on, 'image': image}

dataset_train=SpikesDataset(data_train)
dataset_val=SpikesDataset(data_val)
dataset_test=SpikesDataset(data_test)

train_loader=DataLoader(dataset=dataset_train,batch_size=4,shuffle=True,num_workers=0)
val_loader=DataLoader(dataset=dataset_val,batch_size=4,shuffle=False,num_workers=0)
test_loader=DataLoader(dataset=dataset_test,batch_size=4,shuffle=False,num_workers=0)



In [8]:
# Run this one cell to inspect shapes before training
batch = next(iter(train_loader))
print("spikes_on shape:", batch['spikes_on'].shape)
print("image shape:    ", batch['image'].shape)

spikes_on shape: torch.Size([4, 51, 1, 40, 40])
image shape:     torch.Size([4, 1, 1, 28, 28])


In [9]:
class NeuralDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.decoder = ConvLSTM(
            img_size=(40, 40),
            input_dim=1,
            hidden_dim=2,
            kernel_size=(3, 3),
            cnn_dropout=0.5,
            rnn_dropout=0.5,
            batch_first=True,
            bias=True,
            peephole=False,
            layer_norm=False,
            return_sequence=False,
            bidirectional=False
        )
        self.proj = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, padding=1),  
            nn.ReLU(),
            nn.Conv2d(16, 1, kernel_size=3, padding=1),   
            nn.Upsample(size=(28, 28), mode='bilinear', align_corners=False)  
        )

    def forward(self, x):
        _, last_state, _ = self.decoder(x)
        return self.proj(last_state)  # (B, 1, 28, 28)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
from torch.utils.data import Dataset, DataLoader

class SpikesDataset(Dataset):
    def __init__(self, data_dict):
        self.spikes_on = data_dict['spikes_on']
        self.images    = data_dict['images']

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        spikes_on = self.spikes_on[idx].float().unsqueeze(1)  # no clone
        image     = self.images[idx].float().unsqueeze(0)
        return {'spikes_on': spikes_on, 'image': image}

dataset_train=SpikesDataset(data_train)
dataset_val=SpikesDataset(data_val)
dataset_test=SpikesDataset(data_test)

train_loader=DataLoader(dataset=dataset_train,batch_size=4,shuffle=True,num_workers=0)
val_loader=DataLoader(dataset=dataset_val,batch_size=4,shuffle=False,num_workers=0)
test_loader=DataLoader(dataset=dataset_test,batch_size=4,shuffle=False,num_workers=0)


# ── Training + Validation Pipeline ────────────────────────────────────────────
class NeuralDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.decoder = ConvLSTM(
            img_size=(40, 40),
            input_dim=1,
            hidden_dim=2,
            kernel_size=(3, 3),
            cnn_dropout=0.5,
            rnn_dropout=0.5,
            batch_first=True,
            bias=True,
            peephole=False,
            layer_norm=False,
            return_sequence=False,
            bidirectional=False
        )
        self.proj = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, padding=1),  
            nn.ReLU(),
            nn.Conv2d(16, 1, kernel_size=3, padding=1),   
            nn.Upsample(size=(28, 28), mode='bilinear', align_corners=False)  
        )

    def forward(self, x):
        _, last_state, _ = self.decoder(x)
        return self.proj(last_state)  # (B, 1, 28, 28)
from tqdm import tqdm

def train_one_epoch(model, loader, optimizer, criterion, device, epoch):
    model.train()
    total_loss = 0.0
    loop = tqdm(loader, desc=f"Train epoch {epoch}", leave=True)
    for batch in loop:
        spikes_on = batch['spikes_on'].to(device)
        images    = batch['image'].to(device)
        optimizer.zero_grad(set_to_none=True)  # ← frees grad memory faster

        out  = model(spikes_on)
        loss = criterion(out.squeeze(1), images.squeeze(1).float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
        del spikes_on, images, out, loss  # ← explicit free
        torch.cuda.empty_cache()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device, epoch):
    model.eval()
    total_loss = 0.0
    loop = tqdm(loader, desc=f"Val   epoch {epoch}", leave=True)
    for batch in loop:
        spikes_on = batch['spikes_on'].to(device)
        images    = batch['image'].to(device)

        out  = model(spikes_on)
        loss = criterion(out.squeeze(1), images.squeeze(1).float())
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    return total_loss / len(loader)


def train(model, train_loader, val_loader, optimizer, criterion,
          device, num_epochs=50, save_path="best_model.pt"):
    model.to(device)
    best_val_loss = float('inf')
    history = {'train': [], 'val': []}

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, epoch)
        val_loss   = evaluate(model, val_loader, criterion, device, epoch)

        history['train'].append(train_loss)
        history['val'].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss':             best_val_loss,
            }, save_path)
            tag = "  ✓ saved"
        else:
            tag = ""

        print(f"Epoch [{epoch:>3}/{num_epochs}]  "
              f"train_loss: {train_loss:.6f}  "
              f"val_loss: {val_loss:.6f}{tag}")

    print(f"\nBest val loss: {best_val_loss:.6f}  →  checkpoint: {save_path}")
    return history
# ── Plotting: ground-truth image vs reconstruction ────────────────────────────

@torch.no_grad()
def plot_reconstructions(model, test_loader, device,
                         num_samples=5, checkpoint_path=None):
    """
    Loads the best checkpoint (optional), runs inference on the test set,
    and plots the first `num_samples` pairs of (image, reconstruction).
    """
    if checkpoint_path and os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f"Loaded checkpoint from epoch {ckpt['epoch']} "
              f"(val_loss={ckpt['val_loss']:.6f})")

    model.to(device)
    model.eval()

    # ── Collect samples ───────────────────────────────────────────────────────
    images_all, recons_all = [], []
    for batch in test_loader:
        spikes_on = batch['spikes_on'].to(device)
        images    = batch['image']               # keep on CPU for plotting

        out = model(spikes_on).cpu()             # (B, 1, H, W)

        images_all.append(images.squeeze(1))     # (B, H, W)
        recons_all.append(out.squeeze(1))        # (B, H, W)

        if sum(x.shape[0] for x in images_all) >= num_samples:
            break

    images_all = torch.cat(images_all)[:num_samples]   # (N, H, W)
    recons_all = torch.cat(recons_all)[:num_samples]   # (N, H, W)

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(num_samples, 2,
                             figsize=(5, 2.5 * num_samples),
                             squeeze=False)

    for i in range(num_samples):
        image = images_all[i].float().numpy()
        recon = recons_all[i].numpy()

        vmin = min(image.min(), recon.min())
        vmax = max(image.max(), recon.max())

        axes[i][0].imshow(image, cmap='gray', vmin=vmin, vmax=vmax)
        axes[i][0].set_title(f"Sample {i+1} — image", fontsize=9)
        axes[i][0].axis('off')

        axes[i][1].imshow(recon, cmap='gray', vmin=vmin, vmax=vmax)
        axes[i][1].set_title(f"Sample {i+1} — Reconstruction", fontsize=9)
        axes[i][1].axis('off')

    plt.suptitle("image vs Reconstruction (test set)", fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()
    return fig




In [11]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model     = NeuralDecoder()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = train(
    model, train_loader, val_loader,
    optimizer, criterion,
    device     = device,
    num_epochs = 50,
    save_path  = "best_model.pt"
)



(3, 3)


Train epoch 1:   0%|          | 0/12000 [00:00<?, ?it/s]


TypeError: conv2d() received an invalid combination of arguments - got (list, Parameter, Parameter, tuple, tuple, tuple, int), but expected one of:
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, tuple of ints padding = 0, tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!list of [Tensor, Tensor]!, !Parameter!, !Parameter!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, !int!)
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, str padding = "valid", tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!list of [Tensor, Tensor]!, !Parameter!, !Parameter!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, !int!)


In [ ]:
print("kernel alive")

In [ ]:
fig = plot_reconstructions(
    model, test_loader,
    device           = device,
    num_samples      = 5,
    checkpoint_path  = "best_model.pt"
)